In [66]:
import xgboost as xgb
import pandas as pd
import numpy as np
from sklearn.ensemble import GradientBoostingRegressor, AdaBoostClassifier
from sklearn.model_selection import train_test_split, GridSearchCV

In [67]:
live_avg = pd.read_csv("../data/samples/live_avg.csv")
live_avg_norm = pd.read_csv("../data/samples/live_avg_norm.csv")
live_avg_scaled = pd.read_csv("../data/samples/live_avg_scaled.csv")

In [68]:
to_keep = ['GPU Utilization (%)',
 'Power Draw (Watts)',
 'GPU Current Clock (MHz)',
 'Memory Allocation Used (MB)',
 'Current Time']
to_drop = ['Memory Utilization (%)', 'Time Delta', 'Iteration', 'GPU Clock Utilization', 'GPU Temp (°C)',]

In [69]:
y = live_avg.iloc[:, -2:-1]
y = y['Resonse Time'].tolist()
live_avg = live_avg.iloc[:, :-1]

In [70]:
live_avg_norm = live_avg_norm.drop(columns=['Unnamed: 0'])
live_avg = live_avg.drop(columns=['Unnamed: 0'])
live_avg_scaled = live_avg_scaled.drop(columns=['Unnamed: 0'])

In [71]:
live_avg_norm.columns = live_avg.columns[:-1]
live_avg_norm_aug = live_avg_norm.drop(columns=to_drop)
live_avg_norm = live_avg_norm.drop(columns=['Iteration', 'Current Time'])

In [72]:
X_train, X_test, y_train, y_test = train_test_split(live_avg_norm_aug, y, test_size=0.2, shuffle=True, random_state=20)

In [73]:
training_dmatrix = xgb.DMatrix(X_train, y_train)
testing_dmatrix = xgb.DMatrix(X_test, y_test)

In [62]:
GRID_PARAMS = {
    'xgb':{
        'objective':'reg:squarederror',
        'max_depth':4,
        "learning_rate":0.005,
        "device":"cuda",
        "booster":"gbtree",
        "early_stopping_rounds":100,
        "n_estimators":150,
    }
}

In [103]:
regressor = xgb.XGBRegressor(objective='reg:squarederror', max_depth=4, learning_rate=0.005, device='cpu', booster='gbtree', n_estimators=150)
# cv = GridSearchCV(estimator=regressor, param_grid=GRID_PARAMS['xgb'])

In [ ]:
regressor.fit(X_train, y_train)

In [105]:
regressor.score(X_test, y_test)

0.2479335069656372

In [107]:
dict(zip(live_avg_norm_aug.columns, regressor.feature_importances_))

{'GPU Utilization (%)': 0.38115337,
 'Power Draw (Watts)': 0.060283974,
 'GPU Current Clock (MHz)': 0.5464912,
 'Memory Allocation Used (MB)': 0.012071434,
 'Current Time': 0.0}